In [1]:
import sys
import os
import mysql.connector
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)
print(f'Connected to future database: {config["db_future"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example
Connected to new database: dataleap_v5_migration
Connected to future database: DB_FUTURE


In [22]:
import pandas as pd
import pickle

# =========================================================
# 1. EXTRACT DATA DARI DB LAMA
# =========================================================
print("📥 Mengambil data jadwal dari database lama...")
cursor_old.execute("SELECT * FROM jadwal")
data_jadwal_old = cursor_old.fetchall()
df_raw_jadwal = pd.DataFrame(data_jadwal_old)
print(f"Total data asli di DB lama: {len(df_raw_jadwal)} baris")

# Extract tabel jadwal
df_jadwal_lama = pd.read_sql("SELECT * FROM jadwal", db_old)
import pandas as pd
import pickle

# =========================================================
# 2. TRANSFORMASI TABEL: jadwal (seperti kode Anda)
# =========================================================
print("⚡ Melakukan transformasi tabel 'jadwal'...")

# A. Pembersihan metode belajar
def clean_mode_belajar(val):
    if pd.isna(val) or not str(val).strip():
        return 'Offline'
    s = str(val).strip().capitalize()
    if s in ('Online', 'Offline', 'Hybrid'):
        return s
    return 'Offline'

# B. Pembersihan status arsip
def clean_status_arsip(val):
    try:
        if pd.isna(val):
            return 0
        return int(float(val))
    except:
        return 0

# C. Saring (filter out) jadwal percobaan
df_jadwal_clean = df_raw_jadwal[~df_raw_jadwal['idperiode'].isin(['P00094', 'P00104'])].copy()
skipped_count = len(df_raw_jadwal) - len(df_jadwal_clean)
print(f"ℹ️ Menyaring {skipped_count} data jadwal percobaan. Sisa data: {len(df_jadwal_clean)} baris")

# Siapkan dataframe untuk insert (tanpa id)
df_jadwal_insert = pd.DataFrame({
    'id_kursus': df_jadwal_clean['idpendkursus'],
    'id_periode': df_jadwal_clean['idperiode'],
    'id_level': df_jadwal_clean['idlevel'],
    'id_sesi': df_jadwal_clean['idsesi'],
    'metode_belajar_jadwal': df_jadwal_clean['mode_belajar'].apply(clean_mode_belajar),
    'nama_rombel': df_jadwal_clean['groupwa'].fillna('').str.strip(),
    'status_arsip': df_jadwal_clean['status_archive'].apply(clean_status_arsip),
    'tempat': df_jadwal_clean['tempat'].fillna('Ruang Kelas').replace('', 'Ruang Kelas').str.strip()
})

# Simpan urutan old_id (sesuai urutan df_jadwal_insert)
old_id_list = df_jadwal_clean['idjadwal'].tolist()

# =========================================================
# BUAT ID BARU (SIMULASI AUTO INCREMENT) & MAPPING
# =========================================================
print("🔢 Membuat ID baru (auto increment simulasi)...")
# Tambahkan kolom 'id' dengan angka urut mulai dari 1
df_jadwal_insert.insert(0, 'id', range(1, len(df_jadwal_insert) + 1))

# Buat mapping old -> new
mapping_id_jadwal = dict(zip(old_id_list, df_jadwal_insert['id'].tolist()))
print(f"✅ Mapping ID jadwal selesai. Jumlah: {len(mapping_id_jadwal)}")

# Simpan mapping ke file pickle (opsional)
with open('mapping_id_jadwal.pkl', 'wb') as f:
    pickle.dump(mapping_id_jadwal, f)
print("💾 Mapping disimpan ke 'mapping_id_jadwal.pkl'")

# =========================================================
# 3. TRANSFORMASI TABEL BARU: jadwal_hari (menggunakan mapping)
# =========================================================
print("⚡ Memisahkan kolom 'hari' ke tabel 'jadwal_hari'...")

hari_rows = []
for idx, row in df_jadwal_clean.iterrows():
    old_id = row['idjadwal']
    hari_string = row['hari']
    if pd.isna(hari_string) or not str(hari_string).strip():
        continue
    for hari in [h.strip() for h in hari_string.split(',') if h.strip()]:
        hari_rows.append({
            'id_jadwal': old_id,   # masih old id
            'nama_hari': hari
        })

df_jadwal_hari = pd.DataFrame(hari_rows)

# Ganti id_jadwal dengan ID baru menggunakan mapping
df_jadwal_hari['id_jadwal'] = df_jadwal_hari['id_jadwal'].map(mapping_id_jadwal)
# Hapus baris yang tidak punya mapping (jika ada)
df_jadwal_hari = df_jadwal_hari.dropna(subset=['id_jadwal'])
df_jadwal_hari['id_jadwal'] = df_jadwal_hari['id_jadwal'].astype(int)

print(f"✓ Tabel 'jadwal_hari' siap. Shape: {df_jadwal_hari.shape}")

# =========================================================
# 4. SIMPAN SEMUA DATA UNTUK TAHAP SELANJUTNYA
# =========================================================
fase_4_afrida = {
    'jadwal': df_jadwal_insert,          # sudah punya id baru
    'jadwal_old_ids': old_id_list,       # urutan lama (untuk referensi)
    'jadwal_hari': df_jadwal_hari,       # sudah pakai id baru
    # nanti tambahkan 'jadwal_detail', 'jadwal_pengajar', 'jadwal_siswa', dll.
}

📥 Mengambil data jadwal dari database lama...
Total data asli di DB lama: 551 baris
⚡ Melakukan transformasi tabel 'jadwal'...
ℹ️ Menyaring 2 data jadwal percobaan. Sisa data: 549 baris
🔢 Membuat ID baru (auto increment simulasi)...
✅ Mapping ID jadwal selesai. Jumlah: 549
💾 Mapping disimpan ke 'mapping_id_jadwal.pkl'
⚡ Memisahkan kolom 'hari' ke tabel 'jadwal_hari'...
✓ Tabel 'jadwal_hari' siap. Shape: (975, 2)


In [23]:
display(df_jadwal_insert.head())

,id,id_kursus,id_periode,id_level,id_sesi,metode_belajar_jadwal,nama_rombel,status_arsip,tempat
0,1,K00001,P00006,L00017,S00002,Online,01 GOGO 3B SR2 (ERICA),1,Ruang Kelas 4
1,2,K00001,P00006,L00024,S00002,Offline,02 SO 1C SR2 (QORIN),1,Ruang Kelas 5
2,3,K00001,P00006,L00023,S00001,Offline,03 SO 1B SR1 (TATIK),1,Ruang Kelas 1
3,4,K00001,P00006,L00025,S00003,Offline,04 SO 2A SR3 (TATIK),1,Ruang Kelas 1
4,5,K00001,P00006,L00014,S00003,Offline,05 GOGO 1B SelK3 (ERICA),1,Ruang Kelas 4


In [24]:
# =========================================================
# 4. TRANSFORMASI TABEL: jadwal_detail (sumber: jadwal_detil)
# =========================================================
print("⚡ Melakukan transformasi tabel 'jadwal_detail'...")

# Ambil data mentah
df_detil_lama = pd.read_sql("SELECT * FROM jadwal_detil", db_old)
print(f"  Data mentah: {len(df_detil_lama)} baris")

# Filter hanya idjadwal yang valid (yang lolos filter di jadwal)
valid_jadwal_ids = set(df_jadwal_clean['idjadwal'])
df_detil_lama = df_detil_lama[df_detil_lama['idjadwal'].isin(valid_jadwal_ids)].copy()
print(f"  Setelah filter idjadwal: {len(df_detil_lama)} baris")

# Simpan old_id detail (urutan akan sama dengan df setelah filter)
old_detail_ids = df_detil_lama['idjadwaldetil'].tolist()

# Buat dataframe untuk insert (tanpa id, akan auto increment)
df_jadwal_detail_insert = pd.DataFrame({
    'judul': df_detil_lama['title'].fillna('').astype(str),
    'deskripsi': df_detil_lama['description'].fillna('').astype(str),
    'url_jadwal_detail': df_detil_lama['url'].fillna('').astype(str),
    'id_jadwal': df_detil_lama['idjadwal'],   # masih old_id
    'label_warna': df_detil_lama['color'].fillna('').astype(str),
    'penanda_mulai': pd.to_datetime(df_detil_lama['start'], errors='coerce').dt.date,
    'penanda_selesai': pd.to_datetime(df_detil_lama['end'], errors='coerce').dt.date,
})

# Kolom tambahan default
df_jadwal_detail_insert['id_mitra'] = None
df_jadwal_detail_insert['id_sesi_override'] = None
df_jadwal_detail_insert['status_detail'] = 'Scheduled'
df_jadwal_detail_insert['source_type'] = 'Generated'
df_jadwal_detail_insert['original_jadwal_detail_id'] = None
df_jadwal_detail_insert['has_operational_data'] = 0
df_jadwal_detail_insert['last_generated_at'] = None
df_jadwal_detail_insert['created_at'] = pd.Timestamp.now()
df_jadwal_detail_insert['updated_at'] = pd.Timestamp.now()

# Cleaning deskripsi & URL
df_jadwal_detail_insert['deskripsi'] = (
    df_jadwal_detail_insert['deskripsi']
    .fillna('')
    .replace('', 'Tidak ada deskripsi')
    .str.strip()
)
df_jadwal_detail_insert.loc[df_jadwal_detail_insert['deskripsi'] == '', 'deskripsi'] = 'Tidak ada deskripsi'

df_jadwal_detail_insert['url_jadwal_detail'] = (
    df_jadwal_detail_insert['url_jadwal_detail']
    .fillna('')
    .replace('-', 'Link belum tersedia')
    .str.strip()
)
df_jadwal_detail_insert.loc[df_jadwal_detail_insert['url_jadwal_detail'] == '', 'url_jadwal_detail'] = 'Link belum tersedia'

print(f"✓ df_jadwal_detail_insert siap. Shape: {df_jadwal_detail_insert.shape}")

# =========================================================
# GANTI id_jadwal dengan ID baru dari mapping
# =========================================================
df_jadwal_detail_insert['id_jadwal'] = df_jadwal_detail_insert['id_jadwal'].map(mapping_id_jadwal)
# Hapus baris yang tidak terpetakan (seharusnya tidak ada)
df_jadwal_detail_insert = df_jadwal_detail_insert.dropna(subset=['id_jadwal'])
df_jadwal_detail_insert['id_jadwal'] = df_jadwal_detail_insert['id_jadwal'].astype(int)

# =========================================================
# BUAT ID BARU UNTUK JADWAL_DETAIL (simulasi auto increment)
# =========================================================
df_jadwal_detail_insert.insert(0, 'id', range(1, len(df_jadwal_detail_insert) + 1))

# Mapping old_detail_id -> new_detail_id
mapping_id_jadwal_detail = dict(zip(old_detail_ids, df_jadwal_detail_insert['id'].tolist()))
print(f"✅ Mapping ID jadwal_detail selesai. Jumlah: {len(mapping_id_jadwal_detail)}")

# =========================================================
# SIMPAN KE DICTIONARY & FILE PICKLE
# =========================================================
fase_4_afrida['jadwal_detail'] = df_jadwal_detail_insert
fase_4_afrida['jadwal_detail_old_ids'] = old_detail_ids

⚡ Melakukan transformasi tabel 'jadwal_detail'...
  Data mentah: 17267 baris
  Setelah filter idjadwal: 17257 baris
✓ df_jadwal_detail_insert siap. Shape: (17257, 16)
✅ Mapping ID jadwal_detail selesai. Jumlah: 17257


In [25]:
# =========================================================
# VERIFIKASI FOREIGN KEY: jadwal_detail -> jadwal
# =========================================================
print("="*70)
print("🔍 MEMERIKSA RELASI FK: df_jadwal_detail_insert -> df_jadwal_insert")
print("="*70)

# Ambil set ID jadwal baru (primary key)
pk_jadwal = set(df_jadwal_insert['id'])

# Set ID jadwal di tabel detail (foreign key)
fk_jadwal_detail = set(df_jadwal_detail_insert['id_jadwal'])

# 1. Cek NULL pada kolom id_jadwal
null_count = df_jadwal_detail_insert['id_jadwal'].isna().sum()
print(f"1. Jumlah nilai NULL pada kolom 'id_jadwal' di detail: {null_count}")
if null_count > 0:
    print("   ⚠️ Ada NULL! Perbaiki mapping.")
else:
    print("   ✅ Tidak ada NULL.")

# 2. Cek apakah semua FK ada di PK
invalid_fk = fk_jadwal_detail - pk_jadwal
print(f"2. Jumlah id_jadwal di detail yang TIDAK ADA di tabel jadwal: {len(invalid_fk)}")
if len(invalid_fk) > 0:
    print(f"   ❌ ID tidak valid (contoh 5): {list(invalid_fk)[:5]}")
else:
    print("   ✅ Semua FK valid.")

# 3. Statistik jumlah data
total_detail = len(df_jadwal_detail_insert)
total_jadwal = len(pk_jadwal)
print(f"3. Total data jadwal_detail: {total_detail}")
print(f"   Total data jadwal: {total_jadwal}")
print(f"   Jumlah unik id_jadwal di detail: {len(fk_jadwal_detail)}")

# 4. Cek orphan (detail tanpa parent) – seharusnya 0
orphan = df_jadwal_detail_insert[~df_jadwal_detail_insert['id_jadwal'].isin(pk_jadwal)]
print(f"4. Jumlah data detail orphan (tanpa parent): {len(orphan)}")
if len(orphan) > 0:
    print("   ❌ Ada orphan! Periksa data berikut:")
    display(orphan[['id', 'id_jadwal', 'judul']].head(10))

# 5. Tampilkan contoh join (beberapa baris) untuk inspeksi visual
print("\n5. Contoh data detail dengan parent-nya (5 baris pertama):")
# Gabungkan dengan jadwal untuk melihat nama rombel atau info lain sebagai sampel
sample_join = df_jadwal_detail_insert[['id', 'id_jadwal', 'judul']].head(5).merge(
    df_jadwal_insert[['id', 'nama_rombel', 'metode_belajar_jadwal']],
    left_on='id_jadwal', right_on='id', how='left'
)
display(sample_join)

print("="*70)
if null_count == 0 and len(invalid_fk) == 0 and len(orphan) == 0:
    print("✅ VERIFIKASI LULUS: Semua FK terhubung dengan benar.")
else:
    print("❌ VERIFIKASI GAGAL: Ada masalah pada relasi FK. Perbaiki sebelum lanjut.")

🔍 MEMERIKSA RELASI FK: df_jadwal_detail_insert -> df_jadwal_insert
1. Jumlah nilai NULL pada kolom 'id_jadwal' di detail: 0
   ✅ Tidak ada NULL.
2. Jumlah id_jadwal di detail yang TIDAK ADA di tabel jadwal: 0
   ✅ Semua FK valid.
3. Total data jadwal_detail: 17257
   Total data jadwal: 549
   Jumlah unik id_jadwal di detail: 549
4. Jumlah data detail orphan (tanpa parent): 0

5. Contoh data detail dengan parent-nya (5 baris pertama):


,id_x,id_jadwal,judul,id_y,nama_rombel,metode_belajar_jadwal
0,1,8,08 SO 2A SelK3 (GETA),8,08 SO 2A SelK3 (GETA),Online
1,2,8,08 SO 2A SelK3 (GETA),8,08 SO 2A SelK3 (GETA),Online
2,3,8,08 SO 2A SelK3 (GETA),8,08 SO 2A SelK3 (GETA),Online
3,4,8,08 SO 2A SelK3 (GETA),8,08 SO 2A SelK3 (GETA),Online
4,5,8,08 SO 2A SelK3 (GETA),8,08 SO 2A SelK3 (GETA),Online


✅ VERIFIKASI LULUS: Semua FK terhubung dengan benar.


In [26]:
# =========================================================
# 5. TRANSFORMASI TABEL: jadwal_pengajar (sumber: jadwal_pengajar)
# =========================================================
print("="*70)
print("⚡ Memproses tabel 'jadwal_pengajar'")
print("="*70)

# 1. Ambil data mentah
df_pengajar_lama = pd.read_sql("SELECT * FROM jadwal_pengajar", db_old)
print(f"Data mentah: {len(df_pengajar_lama)} baris")

# 2. Simpan old_id untuk tracing
df_pengajar_lama['old_id_pengajar'] = df_pengajar_lama['idpengajar'].astype(str)
df_pengajar_lama['old_id_jadwal']   = df_pengajar_lama['idjadwal'].astype(str)

# 3. Filter: hanya yang idjadwal-nya ada di mapping (parent valid)
df_pengajar_lama = df_pengajar_lama[df_pengajar_lama['old_id_jadwal'].isin(mapping_id_jadwal.keys())].copy()
print(f"Setelah filter parent valid: {len(df_pengajar_lama)} baris")

# 4. Ambil id_user (prioritas idusers, fallback ke idguru jika ada)
if 'idusers' in df_pengajar_lama.columns:
    id_user_col = 'idusers'
elif 'idguru' in df_pengajar_lama.columns:
    id_user_col = 'idguru'
else:
    id_user_col = None
    print("⚠️ Kolom id_user tidak ditemukan. Semua id_user akan diisi None.")

# Buat dataframe sementara
df_temp = pd.DataFrame({
    'old_id_pengajar': df_pengajar_lama['old_id_pengajar'],
    'old_id_jadwal':   df_pengajar_lama['old_id_jadwal'],
    'id_user':         df_pengajar_lama[id_user_col].astype(str) if id_user_col else np.nan
})

# Bersihkan id_user: ubah 'nan', 'None', '' menjadi NaN
df_temp['id_user'] = df_temp['id_user'].replace(['nan', 'None', ''], np.nan)
df_temp['id_user'] = df_temp['id_user'].fillna(np.nan)

# Hapus baris yang tidak punya id_user (karena FK ke users)
before_drop_user = len(df_temp)
df_temp = df_temp.dropna(subset=['id_user'])
print(f"Baris tanpa id_user dibuang: {before_drop_user - len(df_temp)}")

# 5. Terapkan mapping id_jadwal
df_temp['id_jadwal'] = df_temp['old_id_jadwal'].map(mapping_id_jadwal)
before_drop_map = len(df_temp)
df_temp = df_temp.dropna(subset=['id_jadwal'])
print(f"Orphan (id_jadwal tidak valid) dibuang: {before_drop_map - len(df_temp)}")

# 6. Inner join dengan df_jadwal_insert untuk memastikan FK valid
df_jadwal_pk = df_jadwal_insert[['id']].copy()
df_temp = df_temp.merge(df_jadwal_pk, left_on='id_jadwal', right_on='id', how='inner')
print(f"Setelah inner join dengan jadwal: {len(df_temp)} baris")
df_temp = df_temp.drop(columns=['id'])  # hapus kolom 'id' hasil merge (karena kita pakai id_jadwal)

# 7. Tambahkan kolom created_at, updated_at
df_temp['created_at'] = pd.Timestamp.now()
df_temp['updated_at'] = pd.Timestamp.now()

# 8. Pilih kolom yang diperlukan (tanpa kolom old)
df_jadwal_pengajar = df_temp[['id_jadwal', 'id_user', 'created_at', 'updated_at']].copy()

# 9. Tambahkan id (auto increment) untuk tabel jadwal_pengajar
df_jadwal_pengajar.insert(0, 'id', range(1, len(df_jadwal_pengajar) + 1))

# (Opsional) Rename kolom 'id' menjadi 'id_jadwal_pengajar' jika skema baru mengharapkan nama itu
# df_jadwal_pengajar = df_jadwal_pengajar.rename(columns={'id': 'id_jadwal_pengajar'})

print(f"✅ jadwal_pengajar siap. Shape: {df_jadwal_pengajar.shape}")

# 10. Simpan ke fase_4_afrida
fase_4_afrida['jadwal_pengajar'] = df_jadwal_pengajar

⚡ Memproses tabel 'jadwal_pengajar'
Data mentah: 641 baris
Setelah filter parent valid: 640 baris
Baris tanpa id_user dibuang: 0
Orphan (id_jadwal tidak valid) dibuang: 0
Setelah inner join dengan jadwal: 640 baris
✅ jadwal_pengajar siap. Shape: (640, 5)


In [27]:
pk_jadwal = set(df_jadwal_insert['id'])
fk_pengajar = set(df_jadwal_pengajar['id_jadwal'])
invalid = fk_pengajar - pk_jadwal

print(f"Total pengajar: {len(df_jadwal_pengajar)}")
print(f"Total jadwal: {len(df_jadwal_insert)}")
print(f"Invalid FK ke jadwal: {len(invalid)}")
if len(invalid) == 0:
    print("✅ Semua FK ke jadwal valid!")
else:
    print("❌ Ada invalid, periksa mapping.")

Total pengajar: 640
Total jadwal: 549
Invalid FK ke jadwal: 0
✅ Semua FK ke jadwal valid!


In [28]:
# =========================================================
# 6. TRANSFORMASI TABEL: jadwal_siswa (sumber: jadwal_siswa)
# =========================================================
print("="*70)
print("⚡ Memproses tabel 'jadwal_siswa'")
print("="*70)

# 1. Ambil data mentah
df_siswa_lama = pd.read_sql("SELECT * FROM jadwal_siswa", db_old)
print(f"Data mentah: {len(df_siswa_lama)} baris")

# 2. Simpan old_id untuk tracing
df_siswa_lama['old_id_siswa'] = df_siswa_lama['idjadwal_siswa'].astype(str)
df_siswa_lama['old_id_jadwal'] = df_siswa_lama['idjadwal'].astype(str)

# 3. Filter: hanya yang idjadwal-nya ada di mapping (parent valid)
df_siswa_lama = df_siswa_lama[df_siswa_lama['old_id_jadwal'].isin(mapping_id_jadwal.keys())].copy()
print(f"Setelah filter parent valid: {len(df_siswa_lama)} baris")

# 4. Buat dataframe sementara
df_temp = pd.DataFrame({
    'old_id_siswa': df_siswa_lama['old_id_siswa'],
    'old_id_jadwal': df_siswa_lama['old_id_jadwal'],
    'id_siswa': df_siswa_lama['idsiswa'].astype(str),
    'tanggal_mulai': pd.to_datetime(df_siswa_lama['tgl_mulai'], errors='coerce').dt.date,
    'tanggal_keluar': pd.to_datetime(df_siswa_lama['tgl_keluar'], errors='coerce').dt.date,
    'tanggal_aktif': pd.to_datetime(df_siswa_lama['tgl_aktif'], errors='coerce').dt.date,
    'tambahan_sesi': pd.to_numeric(df_siswa_lama['tambahan_sesi'], errors='coerce').fillna(0).astype(int),
    'tambahan_keterangan': df_siswa_lama['tambahan_ket'].fillna('Belum ada keterangan').astype(str),
    'status_keluar': pd.to_numeric(df_siswa_lama['is_keluar'], errors='coerce').fillna(0).astype(int)
})

# Hapus baris tanpa id_siswa
before_drop_siswa = len(df_temp)
df_temp = df_temp.dropna(subset=['id_siswa'])
print(f"Baris tanpa id_siswa dibuang: {before_drop_siswa - len(df_temp)}")

# 5. Mapping id_jadwal
df_temp['id_jadwal'] = df_temp['old_id_jadwal'].map(mapping_id_jadwal)
df_temp = df_temp.dropna(subset=['id_jadwal'])

# 6. Inner join dengan df_jadwal_insert untuk memastikan FK valid
df_jadwal_pk = df_jadwal_insert[['id']].copy()
df_temp = df_temp.merge(df_jadwal_pk, left_on='id_jadwal', right_on='id', how='inner')
print(f"Setelah inner join dengan jadwal: {len(df_temp)} baris")
df_temp = df_temp.drop(columns=['id'])

# 7. Tambahkan kolom default untuk skema baru
kolom_baru = {
    'is_acc_rapor': 0,
    'status_ketuntasan': None,
    'catatan_ketuntasan_guru': None,
    'catatan_ketuntasan_admin': None,
    'ketuntasan_diperbarui_oleh': None,
    'ketuntasan_diperbarui_pada': None
}
for col, default_val in kolom_baru.items():
    df_temp[col] = default_val

# 8. Pilih kolom final (tanpa old_id)
df_jadwal_siswa = df_temp[[
    'id_jadwal',
    'id_siswa',
    'tanggal_mulai',
    'tanggal_keluar',
    'tanggal_aktif',
    'tambahan_sesi',
    'tambahan_keterangan',
    'status_keluar',
    'is_acc_rapor',
    'status_ketuntasan',
    'catatan_ketuntasan_guru',
    'catatan_ketuntasan_admin',
    'ketuntasan_diperbarui_oleh',
    'ketuntasan_diperbarui_pada'
]].copy()

# 9. Tambahkan id baru (simulasi auto increment)
df_jadwal_siswa.insert(0, 'id', range(1, len(df_jadwal_siswa) + 1))

# 10. Buat mapping old_id_siswa -> id baru (untuk keperluan jika ada tabel anak)
mapping_id_jadwal_siswa = dict(zip(df_temp['old_id_siswa'], df_jadwal_siswa['id']))

# 11. RENAME kolom 'id' menjadi nama primary key di database baru
# Ganti 'id_jadwal_siswa' dengan nama kolom PK yang sebenarnya di tabel jadwal_siswa
df_jadwal_siswa = df_jadwal_siswa.rename(columns={'id': 'id_jadwal_siswa'})

print(f"✅ jadwal_siswa siap. Shape: {df_jadwal_siswa.shape}")
print(f"   Kolom PK: {df_jadwal_siswa.columns[0]}")

# 12. Simpan ke fase_4_afrida
fase_4_afrida['jadwal_siswa'] = df_jadwal_siswa

⚡ Memproses tabel 'jadwal_siswa'
Data mentah: 3905 baris
Setelah filter parent valid: 3903 baris
Baris tanpa id_siswa dibuang: 0
Setelah inner join dengan jadwal: 3903 baris
✅ jadwal_siswa siap. Shape: (3903, 15)
   Kolom PK: id_jadwal_siswa


In [29]:
pk_jadwal = set(df_jadwal_insert['id'])
fk_siswa = set(df_jadwal_siswa['id_jadwal'])
invalid = fk_siswa - pk_jadwal

print(f"Total siswa: {len(df_jadwal_siswa)}")
print(f"Total jadwal: {len(df_jadwal_insert)}")
print(f"Invalid FK ke jadwal: {len(invalid)}")
if len(invalid) == 0:
    print("✅ Semua FK ke jadwal valid!")
else:
    print("❌ Ada invalid, periksa mapping.")

Total siswa: 3903
Total jadwal: 549
Invalid FK ke jadwal: 0
✅ Semua FK ke jadwal valid!


In [30]:
# =========================================================
# 7. TRANSFORMASI TABEL: catatan_kelas (sumber: catatan_kelas)
# =========================================================
print("="*70)
print("⚡ Memproses tabel 'catatan_kelas'")
print("="*70)

# 1. Ambil data mentah
df_catatan_lama = pd.read_sql("SELECT * FROM catatan_kelas", db_old)
print(f"Data mentah: {len(df_catatan_lama)} baris")

# 2. Simpan old_id untuk tracing
df_catatan_lama['old_id_ck'] = df_catatan_lama['idcatatan_kelas'].astype(str)
df_catatan_lama['old_id_jadwal'] = df_catatan_lama['idjadwal'].astype(str)

# Biarkan old_id_jadwal_detail apa adanya (bisa NaN)
# Jangan konversi ke string dulu agar null tetap null
df_catatan_lama['old_id_jadwal_detail'] = df_catatan_lama['idjadwaldetil']  # biarkan asli

# 3. Filter: hanya yang idjadwal-nya ada di mapping (parent jadwal valid)
df_catatan_lama = df_catatan_lama[df_catatan_lama['old_id_jadwal'].isin(mapping_id_jadwal.keys())].copy()
print(f"Setelah filter parent jadwal valid: {len(df_catatan_lama)} baris")

# 4. Buat dataframe sementara
df_temp = pd.DataFrame({
    'old_id_ck': df_catatan_lama['old_id_ck'],
    'old_id_jadwal': df_catatan_lama['old_id_jadwal'],
    'old_id_jadwal_detail': df_catatan_lama['old_id_jadwal_detail'],
    'catatan_kelas': df_catatan_lama['catatan'].fillna('').astype(str),
    'topik_diskusi': df_catatan_lama['materi_diskusi'].fillna('').astype(str),
    'hasil_konfirmasi': df_catatan_lama['hasil_konfirm'].fillna('').astype(str),
    'tanggal_konfirmasi': pd.to_datetime(df_catatan_lama['tglcek'], errors='coerce'),
})

# 5. Mapping id_jadwal
df_temp['id_jadwal'] = df_temp['old_id_jadwal'].map(mapping_id_jadwal)
df_temp = df_temp.dropna(subset=['id_jadwal'])

# 6. Mapping id_jadwal_detail (jika ada)
def map_detail(old_id):
    if pd.isna(old_id):
        return np.nan
    # old_id mungkin string atau angka, pastikan string untuk lookup di mapping
    return mapping_id_jadwal_detail.get(str(old_id), np.nan)

df_temp['id_jadwal_detail'] = df_temp['old_id_jadwal_detail'].apply(map_detail)

# 7. Hapus baris yang memiliki old_id_jadwal_detail tidak null tetapi mapping-nya gagal (orphan detail)
before_drop_detail = len(df_temp)
# Cek baris yang memiliki old_id_jadwal_detail tidak null dan id_jadwal_detail NaN
df_temp = df_temp[~((df_temp['old_id_jadwal_detail'].notna()) & (df_temp['id_jadwal_detail'].isna()))].copy()
print(f"Baris dengan detail orphan dibuang: {before_drop_detail - len(df_temp)}")

# 8. (Opsional) Pastikan detail yang tidak null ada di df_jadwal_detail_insert (inner join)
if len(df_jadwal_detail_insert) > 0:
    valid_detail_ids = set(df_jadwal_detail_insert['id'])  # internal ID
    df_temp = df_temp[~((df_temp['id_jadwal_detail'].notna()) & (~df_temp['id_jadwal_detail'].isin(valid_detail_ids)))].copy()
    print(f"Setelah filter detail dengan inner join: {len(df_temp)} baris")

# 9. Bersihkan tanggal_konfirmasi: jika null, isi dengan created_at (kita buat dulu)
df_temp['created_at'] = pd.Timestamp.now()
df_temp['tanggal_konfirmasi'] = df_temp['tanggal_konfirmasi'].fillna(df_temp['created_at'])

# 10. Tambahkan kolom id_karyawan (None)
df_temp['id_karyawan'] = None

# 11. Pilih kolom final (tanpa old_id)
df_catatan_kelas = df_temp[[
    'id_jadwal',
    'id_jadwal_detail',
    'catatan_kelas',
    'topik_diskusi',
    'tanggal_konfirmasi',
    'hasil_konfirmasi',
    'id_karyawan'
]].copy()

# 12. Tambahkan id (auto increment) internal
df_catatan_kelas.insert(0, 'id', range(1, len(df_catatan_kelas) + 1))

# 13. Buat mapping old_id_ck -> id baru (untuk keperluan jika ada anak)
mapping_id_catatan_kelas = dict(zip(df_temp['old_id_ck'], df_catatan_kelas['id']))

print(f"✅ catatan_kelas siap. Shape: {df_catatan_kelas.shape}")

# 14. Simpan ke fase_4_afrida
fase_4_afrida['catatan_kelas'] = df_catatan_kelas

⚡ Memproses tabel 'catatan_kelas'
Data mentah: 12797 baris
Setelah filter parent jadwal valid: 12797 baris
Baris dengan detail orphan dibuang: 0
Setelah filter detail dengan inner join: 12797 baris
✅ catatan_kelas siap. Shape: (12797, 8)


In [31]:
pk_jadwal = set(df_jadwal_insert['id'])
fk_jadwal = set(df_catatan_kelas['id_jadwal'])
invalid_jadwal = fk_jadwal - pk_jadwal

pk_detail = set(df_jadwal_detail_insert['id'])
fk_detail = set(df_catatan_kelas[df_catatan_kelas['id_jadwal_detail'].notna()]['id_jadwal_detail'])
invalid_detail = fk_detail - pk_detail

print(f"Invalid FK ke jadwal: {len(invalid_jadwal)}")
print(f"Invalid FK ke detail: {len(invalid_detail)}")
if len(invalid_jadwal)==0 and len(invalid_detail)==0:
    print("✅ Semua FK valid!")
else:
    print("❌ Ada invalid, periksa mapping.")

Invalid FK ke jadwal: 0
Invalid FK ke detail: 0
✅ Semua FK valid!


In [32]:
# =========================================================
# 8. TRANSFORMASI TABEL: catatan_kelas_tag (sumber: catatan_kelas_tag)
# =========================================================
print("="*70)
print("⚡ Memproses tabel 'catatan_kelas_tag'")
print("="*70)

# 1. Ambil data mentah
df_tag_lama = pd.read_sql("SELECT * FROM catatan_kelas_tag", db_old)
print(f"Data mentah: {len(df_tag_lama)} baris")

# 2. Simpan old_id_ck
df_tag_lama['old_id_ck'] = df_tag_lama['idcatatan_kelas'].astype(str)

# 3. Filter: hanya yang old_id_ck-nya ada di mapping catatan_kelas (parent valid)
df_tag_lama = df_tag_lama[df_tag_lama['old_id_ck'].isin(mapping_id_catatan_kelas.keys())].copy()
print(f"Setelah filter parent valid: {len(df_tag_lama)} baris")

# 4. Buat dataframe sementara
df_temp = pd.DataFrame({
    'old_id_ck': df_tag_lama['old_id_ck'],
    'id_topik_diskusi': df_tag_lama['idtagmd'].astype(str),  # biarkan sebagai string (nanti dipetakan)
})

# 5. Mapping id_catatan_kelas
df_temp['id_catatan_kelas'] = df_temp['old_id_ck'].map(mapping_id_catatan_kelas)
df_temp = df_temp.dropna(subset=['id_catatan_kelas'])

# 6. Pastikan FK ke catatan_kelas valid (inner join opsional)
pk_ck = set(df_catatan_kelas['id'])  # internal PK catatan_kelas
df_temp = df_temp[df_temp['id_catatan_kelas'].isin(pk_ck)].copy()

# 7. Hapus old_id_ck
df_temp = df_temp.drop(columns=['old_id_ck'])

# 8. Tambahkan id (auto increment) internal
df_temp.insert(0, 'id', range(1, len(df_temp) + 1))

# 9. Dataframe final
df_catatan_kelas_tag = df_temp[['id', 'id_catatan_kelas', 'id_topik_diskusi']].copy()

print(f"✅ catatan_kelas_tag siap. Shape: {df_catatan_kelas_tag.shape}")

# 10. Simpan ke fase_4_afrida
fase_4_afrida['catatan_kelas_tag'] = df_catatan_kelas_tag

⚡ Memproses tabel 'catatan_kelas_tag'
Data mentah: 999 baris
Setelah filter parent valid: 999 baris
✅ catatan_kelas_tag siap. Shape: (999, 3)


In [33]:
pk_ck = set(df_catatan_kelas['id'])
fk_tag = set(df_catatan_kelas_tag['id_catatan_kelas'])
invalid = fk_tag - pk_ck
print(f"Invalid FK ke catatan_kelas: {len(invalid)}")
print("Aman! ✅" if len(invalid)==0 else "❌")

Invalid FK ke catatan_kelas: 0
Aman! ✅


In [34]:
# =========================================================
# 9. TRANSFORMASI TABEL: catatan_mingguan (sumber: catatan_mingguan)
# =========================================================
print("="*70)
print("⚡ Memproses tabel 'catatan_mingguan'")
print("="*70)

# 1. Ambil data mentah
df_cm_lama = pd.read_sql("SELECT * FROM catatan_mingguan", db_old)
print(f"Data mentah: {len(df_cm_lama)} baris")

# 2. Simpan old_id untuk mapping (jika dibutuhkan nanti)
df_cm_lama['old_id_cm'] = df_cm_lama['idcatatanweek'].astype(str)

# 3. Buat dataframe sementara
df_temp = pd.DataFrame({
    'old_id_cm': df_cm_lama['old_id_cm'],
    'id_user': df_cm_lama['idusers'].astype(str),
    'tanggal_mulai_cm': pd.to_datetime(df_cm_lama['tglawal'], errors='coerce').dt.date,
    'tanggal_selesai_cm': pd.to_datetime(df_cm_lama['tglakhir'], errors='coerce').dt.date,
    'tanggal_verifikasi_cm': pd.to_datetime(df_cm_lama['tglcek'], errors='coerce'),
    'keterangan_cm': df_cm_lama['catatan'].fillna('').astype(str),
    'keputusan_cm': df_cm_lama['hasil_konfirmasi'].fillna('').astype(str),
})

# 4. Hapus baris yang tidak punya id_user (FK ke users)
before_drop_user = len(df_temp)
df_temp = df_temp.dropna(subset=['id_user'])
print(f"Baris tanpa id_user dibuang: {before_drop_user - len(df_temp)}")

# 5. Tambahkan kolom created_at & updated_at
df_temp['created_at'] = pd.Timestamp.now()
df_temp['updated_at'] = pd.Timestamp.now()

# 6. (Opsional) Jika ada kolom lain yang perlu ditambahkan default, lakukan di sini
# Misalnya kolom 'status' atau 'jenis' jika ada di skema baru

# 7. Pilih kolom final (tanpa old_id)
df_catatan_mingguan = df_temp[[
    'id_user',
    'tanggal_mulai_cm',
    'tanggal_selesai_cm',
    'tanggal_verifikasi_cm',
    'keterangan_cm',
    'keputusan_cm',
    'created_at',
    'updated_at'
]].copy()

# 8. Tambahkan id (auto increment) internal
df_catatan_mingguan.insert(0, 'id', range(1, len(df_catatan_mingguan) + 1))

# 9. Buat mapping old_id_cm -> id baru (untuk keperluan jika ada tabel anak)
mapping_id_catatan_mingguan = dict(zip(df_temp['old_id_cm'], df_catatan_mingguan['id']))

print(f"✅ catatan_mingguan siap. Shape: {df_catatan_mingguan.shape}")

# 10. Simpan ke fase_4_afrida
fase_4_afrida['catatan_mingguan'] = df_catatan_mingguan
fase_4_afrida['mapping_id_catatan_mingguan'] = mapping_id_catatan_mingguan

⚡ Memproses tabel 'catatan_mingguan'
Data mentah: 0 baris
Baris tanpa id_user dibuang: 0
✅ catatan_mingguan siap. Shape: (0, 9)


In [35]:
# =========================================================
# PERBAIKAN AKHIR SEBELUM SIMPAN PICKLE CLEAN
# =========================================================
print("\n🔧 Melakukan perbaikan pada tabel bermasalah...")

# 1. PERBAIKAN jadwal_pengajar: hapus kolom created_at & updated_at
if 'jadwal_pengajar' in fase_4_afrida:
    df = fase_4_afrida['jadwal_pengajar']
    for col in ['created_at', 'updated_at']:
        if col in df.columns:
            df = df.drop(columns=[col])
            print(f"  ✔ {col} dihapus dari jadwal_pengajar")
    fase_4_afrida['jadwal_pengajar'] = df

# 2. PERBAIKAN jadwal_siswa: isi tanggal null dengan default (sekarang)
if 'jadwal_siswa' in fase_4_afrida:
    df = fase_4_afrida['jadwal_siswa']
    now = pd.Timestamp.now()  # nilai default
    
    # Kolom yang wajib diisi (NOT NULL)
    for col in ['tanggal_mulai', 'tanggal_aktif']:
        if col in df.columns:
            df[col] = df[col].fillna(now)
            print(f"  ✔ {col} diisi dengan '{now}' untuk yang null")
    
    # Kolom yang boleh NULL (tapi kita isi dengan NULL agar tidak error)
    if 'tanggal_keluar' in df.columns:
        df['tanggal_keluar'] = df['tanggal_keluar'].fillna(pd.NA)
        print(f"  ✔ tanggal_keluar diisi NULL (pd.NA) untuk yang null")
    
    # Pastikan tipe datetime (biarkan apa adanya)
    fase_4_afrida['jadwal_siswa'] = df

print("✅ Perbaikan selesai.\n")


🔧 Melakukan perbaikan pada tabel bermasalah...
  ✔ created_at dihapus dari jadwal_pengajar
  ✔ updated_at dihapus dari jadwal_pengajar
  ✔ tanggal_mulai diisi dengan '2026-06-23 13:15:47.527636' untuk yang null
  ✔ tanggal_aktif diisi dengan '2026-06-23 13:15:47.527636' untuk yang null
  ✔ tanggal_keluar diisi NULL (pd.NA) untuk yang null
✅ Perbaikan selesai.



In [36]:
import pickle

print("🧹 Membersihkan kolom 'id' dari semua DataFrame...")

# Daftar nama tabel yang memiliki kolom 'id' internal
tabel_list = [
    'jadwal',
    'jadwal_hari',
    'jadwal_detail',
    'jadwal_pengajar',
    'jadwal_siswa',
    'catatan_kelas',
    'catatan_kelas_tag',
    'catatan_mingguan'
]

# Hapus kolom 'id' dari setiap DataFrame
for tbl in tabel_list:
    if tbl in fase_4_afrida and isinstance(fase_4_afrida[tbl], pd.DataFrame):
        df = fase_4_afrida[tbl]
        if 'id' in df.columns:
            fase_4_afrida[tbl] = df.drop(columns=['id'])
            print(f"  ✔ {tbl}: kolom 'id' dihapus")
        else:
            print(f"  - {tbl}: tidak ada kolom 'id'")

# Simpan sebagai file pickle final (clean)
output_file = 'fase_4_afrida.pkl'
with open(output_file, 'wb') as f:
    pickle.dump(fase_4_afrida, f)

print(f"\n✅ File '{output_file}' siap untuk di-insert handler.")
print("   (Tidak ada kolom 'id' di dalam DataFrame tabel.)")

🧹 Membersihkan kolom 'id' dari semua DataFrame...
  ✔ jadwal: kolom 'id' dihapus
  - jadwal_hari: tidak ada kolom 'id'
  ✔ jadwal_detail: kolom 'id' dihapus
  ✔ jadwal_pengajar: kolom 'id' dihapus
  - jadwal_siswa: tidak ada kolom 'id'
  ✔ catatan_kelas: kolom 'id' dihapus
  ✔ catatan_kelas_tag: kolom 'id' dihapus
  ✔ catatan_mingguan: kolom 'id' dihapus



✅ File 'fase_4_afrida.pkl' siap untuk di-insert handler.
   (Tidak ada kolom 'id' di dalam DataFrame tabel.)


In [37]:
import pickle
import pandas as pd

# =========================================================
# 1. Tentukan file pickle yang akan di-load
# =========================================================
file_pkl = 'fase_4_afrida.pkl'   # Ganti dengan nama file yang sesuai

try:
    with open(file_pkl, 'rb') as f:
        data = pickle.load(f)
    print(f"✅ Berhasil load {file_pkl}")
    print(f"📌 Key yang tersedia: {list(data.keys())}\n")
except FileNotFoundError:
    print(f"❌ File {file_pkl} tidak ditemukan. Coba gunakan 'fase_4_afrida.pkl'")
    # Jika file tidak ditemukan, coba alternatif
    file_pkl = 'fase_4_afrida.pkl'
    try:
        with open(file_pkl, 'rb') as f:
            data = pickle.load(f)
        print(f"✅ Berhasil load {file_pkl}")
        print(f"📌 Key yang tersedia: {list(data.keys())}\n")
    except FileNotFoundError:
        print("❌ Tidak ada file pickle yang ditemukan.")
        exit()

# =========================================================
# 2. Tampilkan informasi setiap tabel (DataFrame)
# =========================================================
print("="*80)
print("📊 INFORMASI TABEL DI DALAM PICKLE")
print("="*80)

# Filter key yang merupakan DataFrame
df_keys = [k for k, v in data.items() if isinstance(v, pd.DataFrame)]
print(f"\n📋 Jumlah tabel: {len(df_keys)}")

for key in df_keys:
    df = data[key]
    print(f"\n🔹 Nama tabel : {key}")
    print(f"   Shape      : {df.shape}")
    print(f"   Kolom      : {list(df.columns)}")
    print(f"   Jumlah null per kolom:")
    print(df.isnull().sum().to_string())
    print(f"\n   Contoh data (5 baris pertama):")
    display(df.head())   # Jika di Jupyter/Colab
    # Jika tidak pakai display, ganti dengan print(df.head())
    print("-"*80)

# =========================================================
# 3. (Opsional) Tampilkan informasi mapping (jika ada)
# =========================================================
print("\n📌 INFORMASI MAPPING (jika ada):")
mapping_keys = [k for k in data.keys() if k.startswith('mapping_')]
for key in mapping_keys:
    mapping = data[key]
    if isinstance(mapping, dict):
        print(f"   {key} : {len(mapping)} entri")
        # Tampilkan 5 contoh mapping pertama
        contoh = list(mapping.items())[:5]
        print(f"     Contoh: {contoh}")
    else:
        print(f"   {key} : {type(mapping)}")

✅ Berhasil load fase_4_afrida.pkl
📌 Key yang tersedia: ['jadwal', 'jadwal_old_ids', 'jadwal_hari', 'jadwal_detail', 'jadwal_detail_old_ids', 'jadwal_pengajar', 'jadwal_siswa', 'catatan_kelas', 'catatan_kelas_tag', 'catatan_mingguan', 'mapping_id_catatan_mingguan']

📊 INFORMASI TABEL DI DALAM PICKLE

📋 Jumlah tabel: 8

🔹 Nama tabel : jadwal
   Shape      : (549, 8)
   Kolom      : ['id_kursus', 'id_periode', 'id_level', 'id_sesi', 'metode_belajar_jadwal', 'nama_rombel', 'status_arsip', 'tempat']
   Jumlah null per kolom:
id_kursus                0
id_periode               0
id_level                 0
id_sesi                  0
metode_belajar_jadwal    0
nama_rombel              0
status_arsip             0
tempat                   0

   Contoh data (5 baris pertama):


,id_kursus,id_periode,id_level,id_sesi,metode_belajar_jadwal,nama_rombel,status_arsip,tempat
0,K00001,P00006,L00017,S00002,Online,01 GOGO 3B SR2 (ERICA),1,Ruang Kelas 4
1,K00001,P00006,L00024,S00002,Offline,02 SO 1C SR2 (QORIN),1,Ruang Kelas 5
2,K00001,P00006,L00023,S00001,Offline,03 SO 1B SR1 (TATIK),1,Ruang Kelas 1
3,K00001,P00006,L00025,S00003,Offline,04 SO 2A SR3 (TATIK),1,Ruang Kelas 1
4,K00001,P00006,L00014,S00003,Offline,05 GOGO 1B SelK3 (ERICA),1,Ruang Kelas 4


--------------------------------------------------------------------------------

🔹 Nama tabel : jadwal_hari
   Shape      : (975, 2)
   Kolom      : ['id_jadwal', 'nama_hari']
   Jumlah null per kolom:
id_jadwal    0
nama_hari    0

   Contoh data (5 baris pertama):


,id_jadwal,nama_hari
0,1,Senin
1,1,Rabu
2,2,Senin
3,2,Rabu
4,3,Senin


--------------------------------------------------------------------------------

🔹 Nama tabel : jadwal_detail
   Shape      : (17257, 16)
   Kolom      : ['judul', 'deskripsi', 'url_jadwal_detail', 'id_jadwal', 'label_warna', 'penanda_mulai', 'penanda_selesai', 'id_mitra', 'id_sesi_override', 'status_detail', 'source_type', 'original_jadwal_detail_id', 'has_operational_data', 'last_generated_at', 'created_at', 'updated_at']
   Jumlah null per kolom:
judul                            0
deskripsi                        0
url_jadwal_detail                0
id_jadwal                        0
label_warna                      0
penanda_mulai                    0
penanda_selesai                  0
id_mitra                     17257
id_sesi_override             17257
status_detail                    0
source_type                      0
original_jadwal_detail_id    17257
has_operational_data             0
last_generated_at            17257
created_at                       0
updated_at          

...

--------------------------------------------------------------------------------

🔹 Nama tabel : catatan_kelas_tag
   Shape      : (999, 2)
   Kolom      : ['id_catatan_kelas', 'id_topik_diskusi']
   Jumlah null per kolom:
id_catatan_kelas    0
id_topik_diskusi    0

   Contoh data (5 baris pertama):


,id_catatan_kelas,id_topik_diskusi
0,1439,T00003
1,1666,T00006
2,1666,T00006
3,1684,T00012
4,1685,T00012


--------------------------------------------------------------------------------

🔹 Nama tabel : catatan_mingguan
   Shape      : (0, 8)
   Kolom      : ['id_user', 'tanggal_mulai_cm', 'tanggal_selesai_cm', 'tanggal_verifikasi_cm', 'keterangan_cm', 'keputusan_cm', 'created_at', 'updated_at']
   Jumlah null per kolom:
id_user                  0
tanggal_mulai_cm         0
tanggal_selesai_cm       0
tanggal_verifikasi_cm    0
keterangan_cm            0
keputusan_cm             0
created_at               0
updated_at               0

   Contoh data (5 baris pertama):


,id_user,tanggal_mulai_cm,tanggal_selesai_cm,tanggal_verifikasi_cm,keterangan_cm,keputusan_cm,created_at,updated_at


--------------------------------------------------------------------------------

📌 INFORMASI MAPPING (jika ada):
   mapping_id_catatan_mingguan : 0 entri
     Contoh: []
